# Dashboard: the fixtures in a listing photo, and what the AC decision costs

The dashboard has two jobs:

- show which fixtures the model finds in any listing photo;
- show what the costed decision ("does this photo show AC?") costs at any pair of thresholds.

Run it with `uv run python main.py app`, and it serves on http://127.0.0.1:7860. It loads
`weights/best.pt` and the two files in `artifacts/` that `train.ipynb` wrote. It never trains. The
only predictions it makes are on photos you upload and on `samples/idealista/`.

In [ ]:
import json
from pathlib import Path

import gradio as gr
import pandas as pd
import plotly.graph_objects as go
from ultralytics import YOLO

ROOT = Path.cwd()
RESULTS = json.loads((ROOT / "artifacts" / "results.json").read_text())
RECORDS = pd.read_csv(ROOT / "artifacts" / "test_scores.csv").to_dict("records")
CFG, OP = RESULTS["config"], RESULTS["operating_point"]
POSITIVE = " + ".join(CFG["POSITIVE_CLASSES"])
MODEL = YOLO(str(ROOT / "weights" / "best.pt"))
CLASSES = [name.strip().lower().replace(" ", "_") for name in MODEL.names.values()]
IMGSZ, SCORE_CONF = 640, 0.001
BLUE, LIGHT_BLUE, ORANGE, GREY = "#2a78d6", "#cde2fb", "#eb6834", "#52514e"


def pct(x):
    return "-" if x is None or x != x else f"{x:.1%}"

## Cost functions

These are copies of the functions in `train.ipynb` that price a band, so a slider costs a band exactly
the way the sweep did. They are copied rather than imported because running the training notebook
would train.

In [ ]:
def decision_rows(records, classes):
    return [{"truth": "positive" if any(r[f"truth_{c}"] for c in classes) else "negative",
             "score": max(r[f"score_{c}"] for c in classes)} for r in records]


def outcome(score, low, high):
    return "matched" if score >= high else "review" if score >= low else "not matched"


def zone_rates(rows, low, high):
    n = len(rows)
    return (sum(r["score"] < low for r in rows) / n,
            sum(low <= r["score"] < high for r in rows) / n,
            sum(r["score"] >= high for r in rows) / n)


def three_way_cost(rows, low, high):
    pos_miss, pos_review, _ = zone_rates([r for r in rows if r["truth"] == "positive"], low, high)
    _, neg_review, neg_match = zone_rates([r for r in rows if r["truth"] == "negative"], low, high)
    pos_n, neg_n = 1000 * CFG["BASE_RATE"], 1000 * (1 - CFG["BASE_RATE"])
    fn = pos_n * (pos_miss + pos_review * (1 - CFG["REVIEWER_RECALL"]))
    fp = neg_n * (neg_match + neg_review * (1 - CFG["REVIEWER_SPECIFICITY"]))
    reviews = pos_n * pos_review + neg_n * neg_review
    cost = (fn * CFG["COST_FN"] + fp * CFG["COST_FP"] + (pos_n - fn) * CFG["COST_TP"]
            + (neg_n - fp) * CFG["COST_TN"] + reviews * CFG["COST_REVIEW"])
    return {"fn": fn, "fp": fp, "reviews": reviews, "reviewed_share": reviews / 1000, "cost": cost}


def staffing(reviews_per_1000):
    per_day = reviews_per_1000 / 1000 * CFG["PHOTOS_PER_DAY"]
    hours = per_day * CFG["SECONDS_PER_REVIEW"] / 3600
    return {"reviews_per_day": per_day, "moderators": hours / CFG["REVIEWER_HOURS_PER_DAY"]}


ROWS = decision_rows(RECORDS, CFG["POSITIVE_CLASSES"])

## Panels

Each fixture's score is its most confident box, the same rule `train.ipynb` uses. Boxes are drawn at
or above the upper threshold. The table lists everything at or above the lower one.

In [ ]:
FIXTURE_COLUMNS = ["fixture", "confidence", "outcome"]


def detect(image):
    return MODEL.predict(image, conf=SCORE_CONF, imgsz=IMGSZ, verbose=False)[0]


def describe(result, low, high):
    best = {}
    for c, conf in zip(result.boxes.cls.tolist(), result.boxes.conf.tolist()):
        best[CLASSES[int(c)]] = max(best.get(CLASSES[int(c)], 0.0), conf)
    rows = [[c, round(s, 3), outcome(s, low, high)]
            for c, s in sorted(best.items(), key=lambda item: -item[1]) if s >= low]
    # plot() draws in BGR, Gradio wants RGB.
    return result[result.boxes.conf >= high].plot()[..., ::-1], pd.DataFrame(rows, columns=FIXTURE_COLUMNS)


def band_summary(low, high):
    counts = {zone: {f"shows {POSITIVE}": 0, f"no {POSITIVE}": 0} for zone in ("matched", "review", "not matched")}
    for r in ROWS:
        counts[outcome(r["score"], low, high)][f"shows {POSITIVE}" if r["truth"] == "positive" else f"no {POSITIVE}"] += 1
    d = three_way_cost(ROWS, low, high)
    s = staffing(d["reviews"])
    text = (
        f"Per 1,000 photos at a base rate of {CFG['BASE_RATE']:.1%}: **EUR {d['cost']:,.2f}**, "
        f"{d['fn']:.1f} FN, {d['fp']:.1f} FP and {d['reviews']:.1f} reviews. **{d['reviewed_share']:.1%}** "
        f"of photos go to a moderator: {s['reviews_per_day']:,.0f} a day at {CFG['PHOTOS_PER_DAY']:,} "
        f"photos a day, which takes {s['moderators']:.2f} moderators."
    )
    return pd.DataFrame([{"outcome": zone, **c} for zone, c in counts.items()]), text


def cost_figure(low, high):
    single = RESULTS["sweep"]["single"]
    fig = go.Figure(go.Scatter(
        x=[s["cut"] for s in single], y=[s["cost"] for s in single], mode="lines+markers",
        line={"color": BLUE, "width": 2}, hovertemplate="cut %{x}<br>EUR %{y:,.2f} per 1,000<extra></extra>",
    ))
    fig.add_vrect(x0=OP["low"], x1=OP["high"], fillcolor=LIGHT_BLUE, opacity=0.5, line_width=0)
    fig.add_trace(go.Scatter(
        x=[OP["high"]], y=[OP["cost"]], mode="markers+text", marker={"size": 14, "color": ORANGE, "symbol": "star"},
        text=[f"chosen band [{OP['low']}, {OP['high']}): EUR {OP['cost']:,.2f}"], textposition="top center",
        hoverinfo="skip",
    ))
    for x in (low, high):
        fig.add_vline(x=x, line={"color": GREY, "dash": "dot"})
    current = three_way_cost(ROWS, low, high)["cost"]
    fig.update_layout(
        title=f"Line: one cut, no moderator. Shaded: chosen band. Dotted: your band, EUR {current:,.2f}",
        xaxis_title="threshold", yaxis_title="EUR per 1,000 photos", template="plotly_white", showlegend=False,
    )
    return fig


def designs_table():
    return pd.DataFrame([
        {"design": name, "band": f"[{d['low']}, {d['high']})", "reviewed": pct(d["reviewed_share"]),
         "EUR per 1,000": round(d["cost"], 2), "reviews/day": round(d["reviews_per_day"]),
         "moderators": round(d["moderators"], 2)}
        for name, d in RESULTS["sweep"]["designs"].items()
    ])


def baselines_table():
    return pd.DataFrame([
        {"system": b["system"], "cut": b["cut"], "recall": pct(b["recall"]),
         "95% CI": f"[{b['recall_ci'][0]:.0%}, {b['recall_ci'][1]:.0%}]",
         "specificity": pct(b["specificity"]), "precision": pct(b["precision"]), "accuracy": pct(b["accuracy"])}
        for b in RESULTS["baselines"]["systems"]
    ])


def per_class_table():
    vs_coco = {r["class"]: r for r in RESULTS["baselines"]["per_class"]}
    return pd.DataFrame([
        {"fixture": r["class"], "test instances": r["test_instances"], "precision": r["precision"],
         "recall": r["recall"], "AP50": r["ap50"], "AP50-95": r["ap50_95"],
         "photo recall at 0.25": vs_coco[r["class"]]["recall"],
         "COCO photo recall": vs_coco[r["class"]]["coco_recall"], "thin": "thin" if r["thin"] else ""}
        for r in RESULTS["per_class"]["classes"]
    ]).round(3)


def confusion_figure():
    cm = RESULTS["confusion_matrix"]
    fig = go.Figure(go.Heatmap(z=cm["matrix"], x=cm["labels"], y=cm["labels"], colorscale="Blues",
                               hovertemplate="true %{x}<br>predicted %{y}<br>%{z} boxes<extra></extra>"))
    fig.update_layout(title=f"Confusion matrix at confidence {cm['conf']} (box counts)",
                      xaxis_title="true class", yaxis={"title": "predicted class", "autorange": "reversed"},
                      template="plotly_white", height=650)
    return fig


def audit_view():
    d = RESULTS["dataset"]
    s = d["splits"]
    text = (f"**{d['dataset']}**: " + ", ".join(f"{name} {s[name]['images']:,} images" for name in s)
            + f". {d['leaked_sources']} source photos appear in more than one split. "
            "Classes marked thin have fewer than 50 test instances, too few to trust a per-class number.")
    table = pd.DataFrame([
        {"fixture": c, **{f"{name} instances": s[name]["classes"][c]["instances"] for name in s},
         "test photos": s["test"]["classes"][c]["images"],
         "median box share": pct(s["train"]["classes"][c]["median_share"]),
         "thin": "thin" if c in d["thin_test_classes"] else ""}
        for c in d["classes"]
    ])
    return text, table


SAMPLES = [(Path(p["path"]).name, detect(str(ROOT / p["path"]))) for p in RESULTS["idealista"]]


def idealista_view(low, high):
    gallery, tables = [], []
    for name, result in SAMPLES:
        annotated, table = describe(result, low, high)
        gallery.append((annotated, name))
        tables.append(table.assign(photo=name))
    return gallery, pd.concat(tables) if tables else pd.DataFrame(columns=FIXTURE_COLUMNS)

## The dashboard

In [ ]:
def refresh(image, low, high):
    annotated, fixtures = describe(detect(image), low, high) if image is not None else (None, pd.DataFrame(columns=FIXTURE_COLUMNS))
    outcomes, summary = band_summary(low, high)
    return (annotated, fixtures, outcomes, summary, cost_figure(low, high), *idealista_view(low, high))


with gr.Blocks(title="Fixture check") as demo:
    gr.Markdown(
        "# What is in this flat?\n"
        f"Scores at or above the upper threshold are matched, below the lower one not matched, and in "
        f"between a moderator checks. The thresholds start at the band chosen for **{POSITIVE}**."
    )
    with gr.Row():
        low = gr.Slider(0, 1, value=min(OP["low"], 1), step=0.0001, label="Lower threshold: below this, not matched")
        high = gr.Slider(0, 1, value=min(OP["high"], 1), step=0.0001, label="Upper threshold: at or above this, matched")

    with gr.Tab("Photo"):
        with gr.Row():
            image = gr.Image(type="pil", label="Listing photo", sources=["upload"])
            annotated = gr.Image(label="Matched boxes", interactive=False)
        fixtures = gr.Dataframe(label="Fixtures found", interactive=False)

    with gr.Tab("Thresholds and cost"):
        gr.Markdown(f"Test photos by outcome and truth for **{POSITIVE}**, priced at the real base rate.")
        outcomes = gr.Dataframe(label="Test photos: outcome by truth", interactive=False)
        summary = gr.Markdown()
        cost = gr.Plot(label="Cost against threshold")
        gr.Dataframe(value=designs_table(), label="The chosen band against the alternatives", interactive=False)
        gr.Dataframe(value=baselines_table(), label="Baselines on the same test photos", interactive=False)

    with gr.Tab("Per class"):
        overall = RESULTS["per_class"]["overall"]
        gr.Markdown(f"Overall mAP50 {overall['map50']:.3f}, mAP50-95 {overall['map50_95']:.3f}. "
                    "The average hides the classes below.")
        gr.Dataframe(value=per_class_table(), label="Per-class results, with COCO zero-shot", interactive=False)
        gr.Plot(value=confusion_figure(), label="Confusion matrix")

    with gr.Tab("Dataset"):
        audit_text, audit_table = audit_view()
        gr.Markdown(audit_text)
        gr.Dataframe(value=audit_table, label="Instances per class", interactive=False)

    with gr.Tab("Idealista"):
        gr.Markdown("A real Spanish listing photo the model never trained on." if SAMPLES
                    else "No photo yet. Add one to `samples/idealista/` and run `main.py train`.")
        gallery = gr.Gallery(label="Idealista sample", columns=2)
        sample_fixtures = gr.Dataframe(label="Fixtures found", interactive=False)

    inputs = [image, low, high]
    outputs = [annotated, fixtures, outcomes, summary, cost, gallery, sample_fixtures]

    # Keep low at or below high: whichever slider moved wins, and the other follows.
    def on_low(image, low, high):
        high = max(low, high)
        return (high, *refresh(image, low, high))

    def on_high(image, low, high):
        low = min(low, high)
        return (low, *refresh(image, low, high))

    low.release(on_low, inputs, [high, *outputs])
    high.release(on_high, inputs, [low, *outputs])
    image.change(refresh, inputs, outputs)
    demo.load(refresh, inputs, outputs)

demo.launch()